In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [ ]:
from torchvision import transforms

transform = transforms.Compose(
    [
        transforms.Resize(96),
        transforms.ToTensor(),    
    ]
)

In [ ]:
train_dataset = datasets.STL10(root='kaggle/input', split='unlabeled', transform=transform, download=True)

In [ ]:
test_dataset = datasets.STL10(root='kaggle/input', split='test',transform=transform, download=True)

In [ ]:
train_loader = DataLoader(train_dataset, shuffle=True, num_workers=4, batch_size=128)
test_loader = DataLoader(test_dataset, shuffle=False, num_workers=2, batch_size=128)

In [ ]:
image = train_dataset[0]
image[0].shape

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.ConvLayers = nn.Sequential(
            nn.Conv2d(3, 6, 3, stride=2, padding=1),   # [48x48]
            nn.BatchNorm2d(6),
            nn.ReLU(),

            nn.Conv2d(6, 12, 3, stride=2, padding=1),  # [24x24]
            nn.BatchNorm2d(12),
            nn.ReLU(),

            nn.Conv2d(12, 24, 3, stride=2, padding=1), # [12x12]
            nn.BatchNorm2d(24),
            nn.ReLU(),

            nn.Conv2d(24, 48, 3, stride=2, padding=1), # [6x6]
            nn.BatchNorm2d(48),
            nn.ReLU()
        )
    
        # Bottleneck layers for mean and log variance
        self.bottleneck_mu = nn.Sequential(
            nn.Conv2d(48, 16, 3, 1, 1),
            nn.Tanh() # Constrains the output
        )
        self.bottleneck_log_var = nn.Sequential(
            nn.Conv2d(48, 16, 3, 1, 1),
            nn.Tanh() # Constrains the output
        )

    def forward(self, x):
        x = self.ConvLayers(x)
        mu = self.bottleneck_mu(x)
        log_var = self.bottleneck_log_var(x)
        return mu, log_var

In [ ]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()

        self.un_bottleneck = nn.Sequential(
            nn.ConvTranspose2d(in_channels=16, out_channels=48, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU()
        )

        self.UpSamplingLayers = nn.Sequential(
            # 6x6 → 12x12
            nn.ConvTranspose2d(48, 24, kernel_size=2, stride=2),
            nn.BatchNorm2d(24),
            nn.ReLU(),

            # 12x12 → 24x24
            nn.ConvTranspose2d(24, 12, kernel_size=2, stride=2),
            nn.BatchNorm2d(12),
            nn.ReLU(),

            # 24x24 → 48x48
            nn.ConvTranspose2d(12, 6, kernel_size=2, stride=2),
            nn.BatchNorm2d(6),
            nn.ReLU(),
        )

        self.final_layer = nn.Sequential(
            # 48x48 → 96x96
            nn.ConvTranspose2d(6, 3, kernel_size=2, stride=2),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.un_bottleneck(z)
        x = self.UpSamplingLayers(x)
        reconstructed_image = self.final_layer(x)
        return reconstructed_image


In [ ]:
class VariationalAutoEncoder(nn.Module):
    def __init__(self):
        super(VariationalAutoEncoder, self).__init__()
        self.encoder = Encoder() 
        self.decoder = Decoder()

    def reparameterize(self, mu, log_var):
       
        std = torch.exp(0.5 * log_var)
        epsilon = torch.randn_like(std)
        return mu + std * epsilon

    def forward(self, x):

        mu, log_var = self.encoder(x)
        
        z = self.reparameterize(mu, log_var)
        
        reconstruction = self.decoder(z)
    
        return reconstruction, mu, log_var

In [ ]:
NUM_EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 1e-4
BETA_START = 0.0
BETA_END = 1.0
ANNEAL_EPOCHS = 50

In [ ]:
def vae_loss(output, images, log_var, mu, beta):

    reconstruction_loss = F.mse_loss(output, images, reduction='sum')
    
    kl_divergence = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    
    total_loss = reconstruction_loss + beta * kl_divergence
    
    return total_loss, reconstruction_loss, kl_divergence

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VariationalAutoEncoder().to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    'min',            
    factor=0.1,       
    patience=5,      
)

In [ ]:
for epoch in range(NUM_EPOCHS):
  
    if epoch < ANNEAL_EPOCHS:
        beta = BETA_START + (BETA_END - BETA_START) * (epoch / ANNEAL_EPOCHS)
    else:
        beta = BETA_END
    model.train()
    

    running_train_loss = 0.0
    running_train_recon_loss = 0.0
    running_train_kl_loss = 0.0

    for images, _ in train_loader:
        images = images.to(device)
        

        reconstruction, mu, log_var = model(images)
        
        total_loss, recon_loss, kl_loss = vae_loss(reconstruction, images, log_var, mu, beta)
        
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Gradient clipping
        optimizer.step()
        

        running_train_loss += total_loss.item() * images.size(0)
        running_train_recon_loss += recon_loss.item() * images.size(0)
        running_train_kl_loss += kl_loss.item() * images.size(0)
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    epoch_train_recon_loss = running_train_recon_loss / len(train_loader.dataset)
    epoch_train_kl_loss = running_train_kl_loss / len(train_loader.dataset)

    model.eval()
    
    running_val_loss = 0.0
    running_val_recon_loss = 0.0
    running_val_kl_loss = 0.0
    
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            
            reconstruction, mu, log_var = model(images)
    
            total_loss, recon_loss, kl_loss = vae_loss(reconstruction, images, log_var, mu, beta)
            
            running_val_loss += total_loss.item() * images.size(0)
            running_val_recon_loss += recon_loss.item() * images.size(0)
            running_val_kl_loss += kl_loss.item() * images.size(0)
    epoch_val_loss = running_val_loss / len(test_loader.dataset)
    epoch_val_recon_loss = running_val_recon_loss / len(test_loader.dataset)
    epoch_val_kl_loss = running_val_kl_loss / len(test_loader.dataset)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Beta: {beta:.4f}")
    print(f"  Train -> Loss: {epoch_train_loss:.4f}, Recon: {epoch_train_recon_loss:.4f}, KL: {epoch_train_kl_loss:.4f}")
    print(f"  Valid -> Loss: {epoch_val_loss:.4f}, Recon: {epoch_val_recon_loss:.4f}, KL: {epoch_val_kl_loss:.4f}")
    
    # Step the scheduler based on the validation loss
    scheduler.step(epoch_val_loss)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

# --- 5. VISUALIZE RESULTS ON RANDOM SAMPLES ---

# Put the model in evaluation mode
model.eval()

# --- Select Random Images from the Test Dataset ---
# NOTE: Use your validation set for this, not the test set you defined earlier
num_images_to_show = 10
num_samples_in_val = len(test_dataset) # Using validation set

# Generate a list of random indices
random_indices = random.sample(range(num_samples_in_val), num_images_to_show)

# Create a custom batch containing only the randomly selected images
# We get the image tensor (index 0) from the dataset for each random index
random_val_images = torch.stack([test_dataset[i][0] for i in random_indices])
random_val_images = random_val_images.to(device)

# Get the reconstructed images for our random batch
with torch.no_grad():
    # --- CORRECTED LINE ---
    # The model now returns a tuple. We only need the first element for visualization.
    reconstructed_images, mu, log_var = model(random_val_images)

# Move tensors to the CPU for plotting
# --- CORRECTED BLOCK ---
# It's safer to move to CPU first, then convert to numpy.
images_np = random_val_images.cpu().numpy()
outputs_np = reconstructed_images.cpu().numpy()

# --- Plotting ---
# (The plotting code itself doesn't need to change, but I'll include it for completeness)
# Helper function to un-normalize and display an image
def denormalize(tensor_np):
    # Rescale from [-1, 1] to [0, 1]
    tensor_np = tensor_np * 0.5 + 0.5
    # Clamp to ensure values are in [0, 1] range
    tensor_np = np.clip(tensor_np, 0, 1)
    # Reorder channels from (C, H, W) to (H, W, C) for plotting
    return np.transpose(tensor_np, (1, 2, 0))


fig, axes = plt.subplots(nrows=2, ncols=num_images_to_show, figsize=(15, 4))
plt.suptitle("Top: Original Images | Bottom: Reconstructed Images", fontsize=16)

for i in range(num_images_to_show):
    # Display original
    ax_orig = axes[0, i]
    ax_orig.imshow(denormalize(images_np[i]))
    ax_orig.get_xaxis().set_visible(False)
    ax_orig.get_yaxis().set_visible(False)

    # Display reconstruction
    ax_recon = axes[1, i]
    ax_recon.imshow(denormalize(outputs_np[i]))
    ax_recon.get_xaxis().set_visible(False)
    ax_recon.get_yaxis().set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
torch.save(model.state_dict(), "vae_weights.pth")